# 06 — Publier `fr_bci_4d` sur Hugging Face

Ce notebook contrôle le dataset final, crée sa dataset card, crée ou réutilise un dépôt Hugging Face de type **dataset**, puis publie les fichiers dans un seul commit.

Le token Hugging Face n'est jamais écrit dans le notebook. Utilisez une connexion locale existante, la variable d'environnement `HF_TOKEN`, ou saisissez le token de façon masquée lorsque le notebook le demande.

## 1. Configuration

Définissez `PUSH_TO_HUB = True` uniquement lorsque les contrôles locaux sont passés. Pour publier dans une organisation, renseignez `HF_NAMESPACE`; sinon le dépôt sera créé sous votre compte personnel.

In [ ]:
from collections import Counter
from getpass import getpass
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
FINAL_DIR = PROJECT_ROOT / "data" / "03_final"
DATASET_PATH = FINAL_DIR / "fr_bci_4d.jsonl"
MANIFEST_PATH = FINAL_DIR / "fr_bci_4d_manifest.json"
README_PATH = FINAL_DIR / "fr_bci_4d_README.md"
REPO_NAME = "fr_bci_4d"

# Laissez vide pour utiliser automatiquement votre nom d'utilisateur Hugging Face.
HF_NAMESPACE = "MON ORGANISATION HF"
PRIVATE_REPO = True   # Passez à False si le jury doit accéder au dataset publiquement.
PUSH_TO_HUB = True  # Passez à True après avoir vérifié les cellules 2 et 3.
COMMIT_MESSAGE = "Publish fr_bci_4d instruction dataset"

print("Dataset local :", DATASET_PATH)
print("Nom du dépôt  :", REPO_NAME)
print("Dépôt privé   :", PRIVATE_REPO)
print("Publication   :", PUSH_TO_HUB)

Dataset local : /home/vedem_worker/scrapt-bau/data/03_final/fr_bci_4d.jsonl
Nom du dépôt  : fr_bci_4d
Dépôt privé   : True
Publication   : True


## 2. Valider le dataset local avant publication

In [7]:
assert DATASET_PATH.exists(), f"Dataset absent : {DATASET_PATH}. Exécutez d'abord le notebook 05."
assert MANIFEST_PATH.exists(), f"Manifeste absent : {MANIFEST_PATH}."
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
rows = []
with DATASET_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"JSON invalide à la ligne {line_number}") from exc

required_fields = {"id", "source_id", "direction", "source_language", "target_language", "instruction", "input", "output"}
assert len(rows) == 55_020
assert all(required_fields <= set(row) for row in rows)
assert all(row["instruction"] and row["output"] for row in rows)
assert len({row["id"] for row in rows}) == len(rows)
assert len({row["source_id"] for row in rows}) == len(rows)
assert len({(row["instruction"], row["input"], row["output"]) for row in rows}) == len(rows)

direction_counts = Counter(row["direction"] for row in rows)
expected_counts = {"bci_bci": 16_506, "fr_fr": 15_406, "fr_bci": 11_554, "bci_fr": 11_554}
assert dict(direction_counts) == expected_counts

digest = hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest()
assert digest == manifest["output_sha256"], "Le checksum ne correspond pas au manifeste."
print("Lignes          :", len(rows))
print("Répartition     :", dict(direction_counts))
print("Sources uniques :", len({row['source_id'] for row in rows}))
print("Triplets uniques:", len({(row['instruction'], row['input'], row['output']) for row in rows}))
print("SHA-256         :", digest)
print("✓ Dataset local prêt pour publication")

Lignes          : 55020
Répartition     : {'bci_bci': 16506, 'fr_fr': 15406, 'fr_bci': 11554, 'bci_fr': 11554}
Sources uniques : 55020
Triplets uniques: 55020
SHA-256         : d16020ee0daaed0be50c563ccf23c7377d037a01396637414980b1aa67f65057
✓ Dataset local prêt pour publication


## 3. Générer la dataset card

In [8]:
dataset_card = '''---
language:
- fr
- bci
license: apache-2.0
task_categories:
- text-generation
pretty_name: fr_bci_4d
size_categories:
- 10K<n<100K
configs:
- config_name: default
  data_files:
  - split: train
    path: data/fr_bci_4d.jsonl
---

# fr_bci_4d

Dataset d'instructions français–baoulé créé par Lree AI Labs pour l'adaptation conversationnelle d'un modèle bilingue.

## Composition

| Direction | Exemples | Proportion |
|---|---:|---:|
| `bci_bci` | 16 506 | 30 % |
| `fr_fr` | 15 406 | 28 % |
| `fr_bci` | 11 554 | 21 % |
| `bci_fr` | 11 554 | 21 % |
| **Total** | **55 020** | **100 %** |

Chaque source est utilisée exactement une fois. Le dataset ne contient aucun chevauchement de sources ni aucun triplet `instruction/input/output` dupliqué.

## Schéma

- `id`: identifiant unique de la ligne ;
- `source_id`: identifiant de l'exemple français d'origine ;
- `direction`: `bci_bci`, `fr_fr`, `fr_bci` ou `bci_fr` ;
- `source_language` et `target_language`: codes de langue ;
- `instruction`, `input`, `output`: champs d'entraînement de type Alpaca ;
- `source_dataset` et `source_row_index`: informations de provenance.

Le code `bci` désigne le baoulé. Les traductions contenant du baoulé ont été produites avec Google Traduction, puis contrôlées automatiquement pour vérifier leur complétude, leur ordre et leur alignement. Une revue humaine reste recommandée avant un usage en production.

## Source et licence

Dérivé de `jpacifico/French-Alpaca-dataset-Instruct-55K`, attribué à Jonathan Pacifico et distribué sous licence Apache-2.0.
'''
README_PATH.write_text(dataset_card, encoding="utf-8")
print("Dataset card :", README_PATH)
print("✓ Dataset card générée")

Dataset card : /home/vedem_worker/scrapt-bau/data/03_final/fr_bci_4d_README.md
✓ Dataset card générée


## 4. Installer le client et s'authentifier

In [9]:
if not PUSH_TO_HUB:
    print("Publication désactivée. Passez PUSH_TO_HUB à True dans la cellule 1 pour continuer.")
    hf_api = None
    repo_id = None
else:
    try:
        from huggingface_hub import CommitOperationAdd, HfApi, login
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "huggingface_hub"])
        from huggingface_hub import CommitOperationAdd, HfApi, login

    hf_token = os.environ.get("HF_TOKEN") or None
    hf_api = HfApi(token=hf_token)
    try:
        identity = hf_api.whoami()
    except Exception:
        hf_token = getpass("Token Hugging Face avec permission d'écriture : ").strip()
        if not hf_token:
            raise RuntimeError("Aucun token fourni.")
        login(token=hf_token, add_to_git_credential=False)
        hf_api = HfApi(token=hf_token)
        identity = hf_api.whoami()

    namespace = HF_NAMESPACE or identity["name"]
    repo_id = f"{namespace}/{REPO_NAME}"
    print("Compte authentifié :", identity["name"])
    print("Dépôt cible        :", repo_id)
    print("Visibilité         :", "privé" if PRIVATE_REPO else "public")

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-4.0.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 3.6 MB/s eta 0:00:00
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.5 MB)
Using cached anyio-4.15.1-py3-none-any.whl (132 kB)
Using cached filelock-4.0.3-py3-none-any.whl (108 kB)
Usin

/home/vedem_worker/scrapt-bau/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Compte authentifié : JauresN16
Dépôt cible        : Tree-AI-lab/fr_bci_4d
Visibilité         : privé


## 5. Créer le dépôt et publier les fichiers

In [10]:
if not PUSH_TO_HUB:
    print("Aucun fichier envoyé : PUSH_TO_HUB=False.")
    commit_info = None
else:
    repository_url = hf_api.create_repo(
        repo_id=repo_id,
        repo_type="dataset",
        private=PRIVATE_REPO,
        exist_ok=True,
    )
    operations = [
        CommitOperationAdd(path_in_repo="data/fr_bci_4d.jsonl", path_or_fileobj=str(DATASET_PATH)),
        CommitOperationAdd(path_in_repo="fr_bci_4d_manifest.json", path_or_fileobj=str(MANIFEST_PATH)),
        CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=str(README_PATH)),
    ]
    commit_info = hf_api.create_commit(
        repo_id=repo_id,
        repo_type="dataset",
        operations=operations,
        commit_message=COMMIT_MESSAGE,
    )
    print("Dépôt  :", repository_url)
    print("Commit :", commit_info.commit_url)


Processing Files (1 / 1): 100%|██████████| 38.9MB / 38.9MB, 3.11MB/s  



Processing Files (1 / 1): 100%|██████████| 38.9MB / 38.9MB, 3.07MB/s  
New Data Upload: 100%|██████████| 38.9MB / 38.9MB, 3.07MB/s  
Validating: ██████████| 100%            


Dépôt  : https://huggingface.co/datasets/Tree-AI-lab/fr_bci_4d
Commit : https://huggingface.co/datasets/Tree-AI-lab/fr_bci_4d/commit/8d644aee268e6623c6cfd2251ff4d1d5cfa6ca3d


## 6. Vérifier les fichiers publiés

In [11]:
if not PUSH_TO_HUB:
    print("Vérification distante ignorée : aucune publication demandée.")
else:
    remote_files = set(hf_api.list_repo_files(repo_id=repo_id, repo_type="dataset"))
    expected_remote_files = {"README.md", "data/fr_bci_4d.jsonl", "fr_bci_4d_manifest.json"}
    missing_remote_files = expected_remote_files - remote_files
    assert not missing_remote_files, f"Fichiers distants absents : {sorted(missing_remote_files)}"
    dataset_url = f"https://huggingface.co/datasets/{repo_id}"
    print("✓ Publication vérifiée")
    print("Dataset :", dataset_url)
    print("Fichiers :", sorted(expected_remote_files))

✓ Publication vérifiée
Dataset : https://huggingface.co/datasets/Tree-AI-lab/fr_bci_4d
Fichiers : ['README.md', 'data/fr_bci_4d.jsonl', 'fr_bci_4d_manifest.json']
